# Clase 4 — Datos de mercado, series de tiempo y beta

Notebook guía. Ejecute cada bloque lentamente y revise el resultado antes de continuar.

**Profesor:** Orlando Joaqui-Barandica  
**Curso:** Introducción a la programación para analítica financiera

## 0. Librerías

En Google Colab, `yfinance` puede instalarse con la primera línea. Si se trabaja sin Internet, el notebook usa la base simulada de respaldo.

In [ ]:
# En Colab descomente esta linea si hace falta:
# !pip -q install yfinance

import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUTA = 'datos/'
os.makedirs('resultados', exist_ok=True)

## 1. JSON → DataFrame

Una respuesta JSON se parece a diccionarios y listas de Python. El archivo incluido es un ejemplo didáctico, no una respuesta en vivo.

In [ ]:
with open(RUTA + 'respuesta_api_ejemplo.json', encoding='utf-8') as archivo:
    respuesta = json.load(archivo)

respuesta

In [ ]:
precios_api = pd.DataFrame(respuesta['data'])
precios_api['date'] = pd.to_datetime(precios_api['date'])
precios_api = precios_api.set_index('date')
precios_api

## 2. Referencia LSEG Workspace / Refinitiv

Este bloque es **referencial**. Una Desktop Session requiere LSEG Workspace/Eikon abierto en la misma máquina donde corre Python. En Colab se necesita una configuración de plataforma compatible.

In [ ]:
# Ejemplo para un entorno LSEG configurado:
# !pip install lseg-data
# import lseg.data as ld
# ld.open_session()
# hist = ld.get_history(
#     universe='AAPL.O',
#     fields=['TRDPRC_1'],
#     interval='1D',
#     start='2026-01-01',
#     end='2026-09-30'
# )
# ld.close_session()

## 3. Funciones para usar yfinance con respaldo local

In [ ]:
def cargar_respaldo():
    return pd.read_csv(
        RUTA + 'mercado_respaldo_clase4.csv',
        parse_dates=['Date']
    ).set_index('Date')

def descargar_accion(ticker='AAPL', inicio='2025-01-01', fin='2026-10-01'):
    try:
        import yfinance as yf
        datos = yf.download(
            ticker, start=inicio, end=fin,
            auto_adjust=True, progress=False,
            multi_level_index=False
        )
        if datos.empty:
            raise ValueError('La descarga llego vacia.')
        return datos, 'yfinance'
    except Exception as error:
        print('No fue posible usar yfinance:', error)
        base = cargar_respaldo()
        columna = ticker + '_SIM'
        if columna not in base.columns:
            columna = 'AAPL_SIM'
        return pd.DataFrame({'Close': base[columna]}).loc[inicio:fin], 'respaldo_simulado'

def descargar_accion_mercado(ticker='AAPL', inicio='2025-01-01', fin='2026-10-01'):
    try:
        import yfinance as yf
        bruto = yf.download(
            [ticker, '^GSPC'], start=inicio, end=fin,
            auto_adjust=True, progress=False
        )
        if bruto.empty:
            raise ValueError('La descarga llego vacia.')
        precios = bruto['Close'].rename(columns={ticker:'Activo', '^GSPC':'Mercado'})
        return precios[['Activo','Mercado']].dropna(), 'yfinance'
    except Exception as error:
        print('No fue posible usar yfinance:', error)
        base = cargar_respaldo()
        columna = ticker + '_SIM'
        if columna not in base.columns:
            columna = 'AAPL_SIM'
        precios = base[[columna,'MARKET_SIM']].rename(columns={columna:'Activo','MARKET_SIM':'Mercado'})
        return precios.loc[inicio:fin].dropna(), 'respaldo_simulado'

## 4. Descargar y reconocer AAPL

In [ ]:
apple, fuente = descargar_accion('AAPL')
print('Fuente:', fuente)
display(apple.head())
print('Forma:', apple.shape)
print('Tipo de indice:', type(apple.index))

## 5. Retorno, medias móviles y volatilidad

In [ ]:
apple = apple.sort_index()
apple['retorno'] = apple['Close'].pct_change()
apple['media_20'] = apple['Close'].rolling(20).mean()
apple['media_60'] = apple['Close'].rolling(60).mean()
apple['vol_20_anual'] = apple['retorno'].rolling(20).std() * np.sqrt(252)

display(apple.tail())

In [ ]:
apple[['Close','media_20','media_60']].plot(figsize=(11,5), title='AAPL: precio y medias moviles')
plt.ylabel('Precio')
plt.show()

In [ ]:
apple['vol_20_anual'].plot(figsize=(11,4), title='AAPL: volatilidad movil anualizada')
plt.ylabel('Volatilidad')
plt.show()

## 6. Estadística descriptiva

In [ ]:
display(apple['retorno'].describe())

retorno_medio_diario = apple['retorno'].mean()
vol_anual = apple['retorno'].std() * np.sqrt(252)
print('Retorno medio diario:', retorno_medio_diario)
print('Volatilidad anualizada:', vol_anual)

## 7. Acción y mercado

In [ ]:
precios, fuente_beta = descargar_accion_mercado('AAPL')
retornos = precios.pct_change().dropna()
print('Fuente:', fuente_beta)
display(precios.head())
display(retornos.corr())

## 8. Regresión simple y beta

In [ ]:
correlacion = retornos['Activo'].corr(retornos['Mercado'])
beta, alpha = np.polyfit(retornos['Mercado'], retornos['Activo'], 1)
beta_alt = retornos['Activo'].cov(retornos['Mercado']) / retornos['Mercado'].var()
r2 = correlacion ** 2

print('Beta:', beta)
print('Beta alternativo:', beta_alt)
print('Alpha diario:', alpha)
print('Correlacion:', correlacion)
print('R2:', r2)

In [ ]:
x = retornos['Mercado']
y = retornos['Activo']
orden = np.argsort(x.to_numpy())
x_ordenado = x.to_numpy()[orden]

plt.figure(figsize=(8,5))
plt.scatter(x, y, alpha=0.30)
plt.plot(x_ordenado, alpha + beta*x_ordenado)
plt.xlabel('Retorno del mercado')
plt.ylabel('Retorno de AAPL')
plt.title(f'AAPL vs mercado - beta = {beta:.2f}')
plt.show()

## 9. Cierre

Cambie `AAPL` por otra acción y repita el flujo. La conclusión debe hablar de **periodo**, **volatilidad**, **correlación** y **beta**, no de predicciones de precio.